In [92]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

In [93]:

API_URL = "https://jsonplaceholder.typicode.com/users"
TIMEOUT_SECONDS = 10

In [94]:
def get_customers(url: str = API_URL) -> list[dict]:
    """Fetch the customer list. Report failures clearly instead of crashing."""
    timeout = TIMEOUT_SECONDS
    try:
        response = requests.get(url, timeout=timeout)
    except requets.exceptions.Timeout:
        print(f'Request timed out after {timeout} seconds. Check your connection and try again. ')
        return[]
    except requests.exceptions.ConnectionError:
        print(f'Could not connect {url}. Check your internet connection or the URL. ')
        return[]
    except requests.exceptions.RequestException as err:
        print(f'Request failed: {err}')
        return[]
    if response.status_code != 200:
        print(f'Unexpected response from the API: status code {response.status_code}')
        return[]

    try:
        customers = response.json()
    except ValueError:
        print('The API responded, but the data was not valid JSON')
        return[]

    return customers

    if __name__ =='__main__':
        customers = get_customers(API_URL)
        print(f'Retrived {len(customers)} customers')
        if customers:
            print('Top-Level fields:', List(customers[0].keys()))


## Explore nested dictionaries and lists
- Implement ``explore(record, indent)`` to one customer record recursively and print every key with its python type, indented by depth. In you README, list which attributes are nested and how you reach the deepest one. 


In [95]:
def explore(record: dict, indent: int = 0) -> None:
    """Print every key in a (possibly nested) record with its type, indented by depth."""
    pad = ' ' * indent

    if isinstance(record, dict):
        for key, value in record.items():
            print(f'{pad}{key}: {type(value).__name__}')
            if isinstance (value, (dict, list)):
                explore(value, indent +1)
    elif isinstance(record, list):
        for position, item in enumerate (record):
            print(f'{pad}[{position}]: {type(item).__name__}')
            if isinstance(item, (dict, list)):
                explore(item, indent +1)

In [96]:
def _to_float(value):
    '''Convert a coordinate string like '-37.3159' to float, or None if impossible'''
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

## Extract Customer Attributes. 
- Implement ``extract_attributes(customers)`` to return on flat dict per customer with id, name, username, email, email domain, phone, city, zipcode, latitude, longitude, company and website. Latitude and longitude must be floats 

In [97]:

def extract_attributes(customers: list[dict]) -> list[dict]:
    """Return one flat dict per customer with the attributes listed in the task."""
    flat_customers = []
    for customer in customers:
        # .get(.., {}) keeps a missing nested block from raising a  keyError
        address = customer.get('address') or {}
        geo = address.get('geo') or {}
        company = customer.get('company') or {}
        email = customer.get('email') or ''

        flat_customers.append(
            {
                'id': customer.get('id'),
                'name': customer.get('name'),
                'username': customer.get('username'),
                'email': email,
                'email_domain': email.split('@')[-1].lower() if '@' in email else None,
                'phone': customer.get('phone'),
                'city': address.get ('city'),
                'Zipcode': address.get('Zipcode'),
                'latitude': _to_float(geo.get('lat')),
                'longitude': _to_float(geo.get('lng')),
                'company': company.get('name'),
                'website': customer.get('website'),
            }
        )
    return flat_customers

if __name__ == '__main__':
    customer = get_customers(API_URL)
    print(f'Retrived{len(customers)} customers')
    if customers:
        print('\nStructure of the first customer record:')
        explore(customer[0])
        print('\nFlat record for the first customer:')
        for key, value in extract_attributes(customers)[0].items():
            print(f' {key}: {value!r}')

Retrived10 customers

Structure of the first customer record:
id: int
name: str
username: str
email: str
address: dict
 street: str
 suite: str
 city: str
 zipcode: str
 geo: dict
  lat: str
  lng: str
phone: str
website: str
company: dict
 name: str
 catchPhrase: str
 bs: str

Flat record for the first customer:
 id: 1
 name: 'Leanne Graham'
 username: 'Bret'
 email: 'Sincere@april.biz'
 email_domain: 'april.biz'
 phone: '1-770-736-8031 x56442'
 city: 'Gwenborough'
 Zipcode: None
 latitude: -37.3159
 longitude: 81.1496
 company: 'Romaguera-Crona'
 website: 'hildegard.org'


## Use sets to find unique cities. 
- Implement ``unique_cities(customers)`` with a set (a set comprehesion is fine). print how many there are and compare it with the number of customers - expalin in one sentance what the comparison tells marketing

In [98]:
def unique_cities(customers: list[dict]) -> set[str]:
    """Return the set of distinct cities."""
    return {c['city'] for c in customers if c.get('city')}

## Count customers by city with loops and dict comprehensions
- Implement ``city_counts(customers)`` without ``collections.Counter`` or pandas and ``email_domain_counts(customers)`` using a dict comprehension. print both sorted from most to least common. 

In [99]:
def city_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per city using a loop and/or a dict comprehension (no Counter, no pandas)."""
    counts = {}
    for customer in customers:
        city = customer.get('city')
        if city:
            counts[city] = counts.get(city, 0) + 1
    return counts

In [100]:
def report_city_spread(customers: list[dict]) -> None:
    '''Print unique cities vs. customers, with a one-sentence takeaway for marketing. '''
    total = len(customers)
    cities = unique_cities(customers)
    print(f'{len(cities)} unique cities for {total} customers')

    if total == 0:
        return
    if len(cities)  == total:
        print('Takeaway: every customer is in a different city, so customers are spread thin and marketing should plan campaigns by wider region, not by city')
    elif len (cities) < total:
        print('Takeaway: some cities hold several customers, so marketing can run targeted city-level campaigns where customers cluster.')

if __name__ == '__main__':
    customers = get_customers(API_URL)
    print(f'Retrived {len(customers)} customers')
    if customers:
        print('\nStructure of the first customer record:')
        explore(customers[0])

        flat = extract_attributes(customers)
        print('\nFlat record for the first customer:')
        for key, value in flat[0].items():
            print(f'{key}: {value!r}')

        print()
        report_city_spread(flat)
        print('\nCustomers per city (most first): ')
        counts = city_counts(flat)
        for city in sorted(counts, key=lambda c: (-counts[c], c)):
            print(f' {city}: {counts[city]}')


Retrived 10 customers

Structure of the first customer record:
id: int
name: str
username: str
email: str
address: dict
 street: str
 suite: str
 city: str
 zipcode: str
 geo: dict
  lat: str
  lng: str
phone: str
website: str
company: dict
 name: str
 catchPhrase: str
 bs: str

Flat record for the first customer:
id: 1
name: 'Leanne Graham'
username: 'Bret'
email: 'Sincere@april.biz'
email_domain: 'april.biz'
phone: '1-770-736-8031 x56442'
city: 'Gwenborough'
Zipcode: None
latitude: -37.3159
longitude: 81.1496
company: 'Romaguera-Crona'
website: 'hildegard.org'

10 unique cities for 10 customers
Takeaway: every customer is in a different city, so customers are spread thin and marketing should plan campaigns by wider region, not by city

Customers per city (most first): 
 Aliyaview: 1
 Bartholomebury: 1
 Gwenborough: 1
 Howemouth: 1
 Lebsackbury: 1
 McKenziehaven: 1
 Roscoeview: 1
 South Christy: 1
 South Elvis: 1
 Wisokyburgh: 1


In [101]:
def to_dataframe(rows: list[dict]) -> pd.DataFrame:
    """Convert the flat customer dicts into a pandas DataFrame, one row per customer."""
    return pd.DataFrame(rows)

In [102]:
def email_domain_counts(customers: list[dict]) -> dict[str, int]:
    """Count customers per email top-level domain (e.g. 'biz', 'tv')."""
    counts = {}
    for customer in customers:
        domain = customer.get("email_domain")
        if domain:
            tld = domain.rsplit(".", 1)[-1]  # last label: "april.biz" -> "biz"
            counts[tld] = counts.get(tld, 0) + 1
    return counts

In [103]:
BAR_COLOR = "#2a78d6"
SURFACE = "#fcfcfb"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
GRID = "#e4e3df"

In [104]:
def plot_distribution(
    counts: dict[str, int],
    path: Path,
    title: str = "Customers per city",
    xlabel: str = "Number of customers",
) -> None:
    """Save a labelled chart of the distribution to path."""
    if not counts:
        print("Nothing to plot: the counts are empty.")
        return

    # Largest first; reversed so the biggest bar sits at the top of a horizontal chart
    items = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    labels = [name for name, _ in items][::-1]
    values = [value for _, value in items][::-1]

    # Width fixed for a slide; height grows with the bar count (capped at 16:9) so bars stay slim
    height = min(5.6, max(3.4, 0.42 * len(items) + 1.9))
    fig, ax = plt.subplots(figsize=(10, height), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    bars = ax.barh(labels, values, height=0.55, color=BAR_COLOR)

    # Value at the tip of each bar, in text colour (never the series colour)
    for bar, value in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.015, bar.get_y() + bar.get_height() / 2,
                f"{value:,}", va="center", ha="left", fontsize=12, color=TEXT_PRIMARY)

    ax.set_xlim(0, max(values) * 1.12)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_title(title, loc="left", fontsize=18, fontweight="bold", color=TEXT_PRIMARY, pad=16)
    ax.set_xlabel(xlabel, fontsize=12, color=TEXT_SECONDARY, labelpad=10)
    ax.tick_params(axis="y", labelsize=12, colors=TEXT_PRIMARY, length=0)
    ax.tick_params(axis="x", labelsize=11, colors=TEXT_SECONDARY, length=0)

    ax.xaxis.grid(True, color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(GRID)

    fig.tight_layout()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200, facecolor=SURFACE)
    plt.close(fig)
    print(f"Chart saved to {path}")



In [105]:
CITY_COUNT_COLUMNS = ["city", "customers"]  # headers of output/city_counts.csv

def export_results(df: pd.DataFrame, counts: dict[str, int], out_dir: Path = Path("output")) -> None:
    """Write customers.csv and city_counts.csv into out_dir, without the pandas index."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    customers_path = out_dir / "customers.csv"
    df.to_csv(customers_path, index=False)

    # Most customers first, ties alphabetical, so the file reads the same on every run
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    city_df = pd.DataFrame(ranked, columns=CITY_COUNT_COLUMNS)
    city_counts_path = out_dir / "city_counts.csv"
    city_df.to_csv(city_counts_path, index=False)

    print(f"Wrote {customers_path} ({len(df)} rows) and {city_counts_path} ({len(city_df)} rows)")


In [106]:
def confirm_counts(df: pd.DataFrame, counts: dict[str, int]) -> bool:
    """Check that pandas' per-city counts match the hand-written `counts` dict."""
    pandas_counts = df["city"].value_counts().to_dict()
    matches = pandas_counts == counts
    print(f"pandas value_counts() agrees with city_counts(): {matches}")
    return matches

In [107]:
if __name__ == "__main__":
    # Step 1: retrieve
    customers = get_customers(API_URL)
    print(f"Retrieved {len(customers)} customers")
    if not customers:
        print("No data to analyse. Fix the connection problem above and run again.")
        raise SystemExit(1)
 
    # Step 2: explore the nested structure
    print("\nStructure of the first customer record:")
    explore(customers[0])
 
    # Step 3: extract the attributes that matter
    flat = extract_attributes(customers)
    print("\nFlat record for the first customer:")
    for key, value in flat[0].items():
        print(f"  {key}: {value!r}")
 
    # Step 4: plain Python answers
    print()
    report_city_spread(flat)
    counts = city_counts(flat)
    print("\nCustomers per city (most first):")
    for city in sorted(counts, key=lambda c: (-counts[c], c)):
        print(f"  {city}: {counts[city]}")
 
    tlds = email_domain_counts(flat)
    print("\nCustomers per email top-level domain:")
    for tld in sorted(tlds, key=lambda t: (-tlds[t], t)):
        print(f"  .{tld}: {tlds[tld]}")
 
    # Step 5: the same answers with pandas
    df = to_dataframe(flat)
    print(f"\nDataFrame shape: {df.shape}")
    print("dtypes:")
    print(df.dtypes)
    print()
    pandas_agrees = confirm_counts(df, counts)
    print(df.groupby("city").size().sort_values(ascending=False))
 
    # Step 6: the chart for the slide
    # Step 6: the charts for the slide
    chart_path = Path("output") / "customer_distribution.png"
    domain_chart_path = Path("output") / "email_domain_distribution.png"
    print()
    plot_distribution(counts, chart_path)
    plot_distribution(
        {f".{tld}": n for tld, n in tlds.items()},  # show as ".biz", ".tv", ...
        domain_chart_path,
        title="Customers per email domain ending",
        xlabel="Number of customers",
    )

    # Step 7: export the results
    export_results(df, counts)
 
    # Findings summary
    top_city, top_n = max(counts.items(), key=lambda kv: (kv[1], kv[0])) if counts else (None, 0)
    top_tld = max(tlds.items(), key=lambda kv: (kv[1], kv[0])) if tlds else None
    print("\n" + "=" * 50)
    print("FINDINGS")
    print("=" * 50)
    print(f"- Customers: {len(customers)}")
    print(f"- Unique cities: {len(unique_cities(flat))}"
          + (" (every customer in a different city)" if len(unique_cities(flat)) == len(flat)
             else f" (largest: {top_city} with {top_n})"))
    if top_tld:
        print(f"- Most common email domain ending: .{top_tld[0]} ({top_tld[1]} customers)")
    print(f"- Table: {df.shape[0]} rows x {df.shape[1]} columns; pandas matches hand counts: {pandas_agrees}")
    print(f"- Charts: {chart_path} (cities), {domain_chart_path} (email domain endings)")
    print("- CSV exports: output/customers.csv, output/city_counts.csv")

Retrieved 10 customers

Structure of the first customer record:
id: int
name: str
username: str
email: str
address: dict
 street: str
 suite: str
 city: str
 zipcode: str
 geo: dict
  lat: str
  lng: str
phone: str
website: str
company: dict
 name: str
 catchPhrase: str
 bs: str

Flat record for the first customer:
  id: 1
  name: 'Leanne Graham'
  username: 'Bret'
  email: 'Sincere@april.biz'
  email_domain: 'april.biz'
  phone: '1-770-736-8031 x56442'
  city: 'Gwenborough'
  Zipcode: None
  latitude: -37.3159
  longitude: 81.1496
  company: 'Romaguera-Crona'
  website: 'hildegard.org'

10 unique cities for 10 customers
Takeaway: every customer is in a different city, so customers are spread thin and marketing should plan campaigns by wider region, not by city

Customers per city (most first):
  Aliyaview: 1
  Bartholomebury: 1
  Gwenborough: 1
  Howemouth: 1
  Lebsackbury: 1
  McKenziehaven: 1
  Roscoeview: 1
  South Christy: 1
  South Elvis: 1
  Wisokyburgh: 1

Customers per email t